# SPT Model Demo
Load a trained model, run inference, visualise and score.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path
import hydra
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path('/cluster/home/larshfle/superpoint_transformer_new')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.utils import init_config
from src.transforms import *
from src.data import *

print('Imports OK')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')


# 0 visualization without GPU

In [ ]:
import sys
sys.path.insert(0, '/cluster/home/larshfle/superpoint_transformer_new')

from src.data import NAG
from src.datasets.norway_combined_3class_config import CLASS_NAMES, CLASS_COLORS, NORWAY_COMBINED_3CLASS_NUM_CLASSES

tilename ='bergen2022_32-1-467-145-17__TILE_3-3_OF_3-3'

H5 = f'/cluster/home/larshfle/datasets/norway_combined_3class/processed/test/999b969bc9e7f45e48797db8a658cff9/{tilename}.h5'

nag = NAG.load(H5)
nag.show(
    class_names=CLASS_NAMES,
    class_colors=CLASS_COLORS,
    #center=[321, -220, 30],
    #radius=20,
    stuff_classes=list(range(NORWAY_COMBINED_3CLASS_NUM_CLASSES)),
    num_classes=NORWAY_COMBINED_3CLASS_NUM_CLASSES,
)


In [ ]:
import sys, torch
sys.path.insert(0, '/cluster/home/larshfle/superpoint_transformer_new')

from src.utils import init_config
from src.data import NAG
from src.datasets.norway_combined_3class_config import CLASS_NAMES, CLASS_COLORS, NORWAY_COMBINED_3CLASS_NUM_CLASSES
    

CKPT = '/cluster/home/larshfle/superpoint_transformer_new/logs/train/runs/2026-05-20_16-26-44/checkpoints/last.ckpt'
H5 = f'/cluster/home/larshfle/datasets/norway_combined_3class/processed/test/999b969bc9e7f45e48797db8a658cff9/{tilename}.h5'
device = 'cpu'


In [ ]:
import hydra
import torch

# Last config og modell
cfg = init_config(overrides=[
    'experiment=semantic/norway_combined_3class',
    'datamodule=semantic/norway_combined_3class',
    f'paths.data_dir=/cluster/home/larshfle/datasets/norway_combined_3class',
])

model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(CKPT, map_location=device)
model = model.eval().to(device)
print("Modell lastet på CPU")




In [ ]:
# Last og transformer NAG
# Last og transformer NAG
datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.setup('fit')
dataset = datamodule.test_dataset

tile = tilename
idx = dataset.cloud_ids.index(tile)
nag = dataset[idx]
nag = dataset.on_device_transform(nag.to(device))
print(f"Punkter: {nag[0].pos.shape[0]}, Superpoints: {nag[1].pos.shape[0]}")



In [ ]:
# Inference
with torch.no_grad():
    output = model(nag)

nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)
print("Ferdig!")


In [ ]:
# Visualiser GT
nag.show(class_names=CLASS_NAMES, class_colors=CLASS_COLORS,
         #center=[405, -197, 44], radius=30,
         stuff_classes=list(range(NORWAY_COMBINED_3CLASS_NUM_CLASSES)),
         num_classes=NORWAY_COMBINED_3CLASS_NUM_CLASSES, title='Ground truth')


In [ ]:
# Visualiser prediksjon
nag.show(class_names=CLASS_NAMES, class_colors=CLASS_COLORS,
         stuff_classes=list(range(NORWAY_COMBINED_3CLASS_NUM_CLASSES)),
         num_classes=NORWAY_COMBINED_3CLASS_NUM_CLASSES,
         title='Prediction', pred=True)


In [ ]:
from src.datasets.norway_combined_3class_config import NORWAY_COMBINED_3CLASS_NUM_CLASSES
import pandas as pd

gt   = nag[0].y.argmax(dim=-1).cpu().numpy()
pred = nag[0].semantic_pred.cpu().numpy()
valid = gt < NORWAY_COMBINED_3CLASS_NUM_CLASSES

rows = []
for c, name in enumerate(dataset.class_names[:dataset.num_classes]):
    tp = int(((gt[valid]==c) & (pred[valid]==c)).sum())
    fp = int(((gt[valid]!=c) & (pred[valid]==c)).sum())
    fn = int(((gt[valid]==c) & (pred[valid]!=c)).sum())
    iou  = tp/(tp+fp+fn) if (tp+fp+fn)>0 else float("nan")
    prec = tp/(tp+fp)    if (tp+fp)>0    else float("nan")
    rec  = tp/(tp+fn)    if (tp+fn)>0    else float("nan")
    rows.append({"class": name, "IoU": f"{100*iou:.1f}%",
                 "Precision": f"{100*prec:.1f}%", "Recall": f"{100*rec:.1f}%",
                 "GT pts": f"{tp+fn:,}"})

display(pd.DataFrame(rows).set_index("class"))


## 1. Settings
Change these to switch model/dataset.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Model ────────────────────────────────────────────────────────────────
EXPERIMENT  = 'experiment=semantic/bro'
CKPT_PATH   = str(PROJECT_ROOT / 'logs/train/runs/2026-05-06_19-01-24/checkpoints/epoch_399.ckpt')
DATA_DIR    = '/cluster/home/larshfle/datasets/bro'
SPLIT       = 'test'
TILE        = 'bro_001709'

# ── Visualisation ────────────────────────────────────────────────────────
MAX_POINTS  = 150_000
CROP_RADIUS = None   # set to e.g. 30 for a zoomed crop
SHOW_PRED   = True   # True = predictions, False = ground truth


## 2. Load config & datamodule

In [ ]:
cfg = init_config(overrides=[
    EXPERIMENT,
    f"ckpt_path={CKPT_PATH}",
    "datamodule.mini=false",
    "datamodule.load_full_res_idx=true",
])
cfg.datamodule.data_dir = DATA_DIR

datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.prepare_data()
datamodule.setup()
dataset = getattr(datamodule, f"{SPLIT}_dataset")

sample_idx = next(i for i, cid in enumerate(dataset.cloud_ids) if cid.startswith(TILE))
print(f'Tile: {dataset.cloud_ids[sample_idx]}')
dataset.print_classes()


## 3. Load model

In [ ]:
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(CKPT_PATH)
model = model.eval().to(device)
model.net.store_features = True
print('Model loaded.')


## 4. Inference

In [ ]:
nag = dataset[sample_idx]
nag = dataset.on_device_transform(nag.to(device))

with torch.no_grad():
    output = model(nag)

nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)
print(f'Points: {nag[0].pos.shape[0]:,}  |  Superpoints: {nag[1].pos.shape[0]:,}')


## 5. Scores

In [ ]:
from src.datasets.bro_config import BRO_NUM_CLASSES

gt   = nag[0].y.argmax(dim=-1).cpu().numpy()
pred = nag[0].semantic_pred.cpu().numpy()
valid = gt < BRO_NUM_CLASSES

rows = []
for c, name in enumerate(dataset.class_names[:dataset.num_classes]):
    tp = int(((gt[valid]==c) & (pred[valid]==c)).sum())
    fp = int(((gt[valid]!=c) & (pred[valid]==c)).sum())
    fn = int(((gt[valid]==c) & (pred[valid]!=c)).sum())
    iou  = tp/(tp+fp+fn) if (tp+fp+fn)>0 else float("nan")
    prec = tp/(tp+fp)    if (tp+fp)>0    else float("nan")
    rec  = tp/(tp+fn)    if (tp+fn)>0    else float("nan")
    rows.append({"class": name, "IoU": f"{100*iou:.1f}%",
                 "Precision": f"{100*prec:.1f}%", "Recall": f"{100*rec:.1f}%",
                 "GT pts": f"{tp+fn:,}"})

display(pd.DataFrame(rows).set_index("class"))


## 6. Visualisation

In [ ]:
show_kwargs = dict(
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=MAX_POINTS,
    semantic_pred=SHOW_PRED,
)
if CROP_RADIUS:
    show_kwargs["radius"] = CROP_RADIUS
    show_kwargs["center"] = nag[0].pos.mean(dim=0).view(1,-1)

nag.show(**show_kwargs)


## 7. Export HTML

In [ ]:
out = PROJECT_ROOT / f'visualizations/demo_{TILE}.html'
out.parent.mkdir(exist_ok=True)
nag.show(
    figsize=1600, path=str(out), display=False,
    title=f"{TILE}  |  {"Predictions" if SHOW_PRED else "Ground Truth"}",
    **show_kwargs,
)
print(f'Saved: {out}')
